# Advanced Python: Raising Exceptions — 27 Problems with Complete Solutions

**Topic from the uploaded lesson:** the `raise` statement; exception classes and instances; `BaseException.args`; bare re-raising; exception translation; implicit/explicit chaining; `raise ... from None`; custom exceptions and tracebacks; boolean-conversion examples.

**Format:** 27 progressively challenging problems, worked solutions, executable assertions, and real-world capstones. Every code cell is intended to run in order without leaving an uncaught intentional exception. Dependencies: Python standard library only. Core exercises support Python 3.10+; the optional `add_note` demonstration checks availability at runtime.

**How to study:** read each problem, attempt your solution in a new cell, compare with the worked solution, then run the following test cell. Use *Kernel → Restart & Run All* for an end-to-end check.

**Important corrections to the source notes:**
1. `raise ValueError` **is legal**: Python instantiates an exception class that derives from `BaseException`. Raising an existing instance is also legal.
2. `raise NewError(...) from None` suppresses *display* of implicit context; it does **not** delete the underlying `__context__` or the new exception's own traceback.
3. `raise NewError(...) from old_error` sets the explicit **cause**, not the new exception's traceback. Use `.with_traceback(tb)` to deliberately attach a traceback (rarely required); even then, new raise frames appear.
4. Custom application errors generally inherit from `Exception`, not directly from `BaseException`. Catch specific failures, avoid `except BaseException`, and do not accidentally swallow cancellation/interrupt signals.

**Scope:** The first sections follow and deepen the supplied lesson. Logging, cleanup, retry, batch ingestion, and ExceptionGroup are explicitly marked as applied extensions.

In [1]:
import sys
import traceback
from io import StringIO


def expect_exception(exception_type, action, *, contains=None):
    """Run zero-argument action and return the expected exception."""
    try:
        action()
    except exception_type as error:
        if contains is not None:
            assert contains in str(error), (contains, str(error))
        return error
    else:
        raise AssertionError(f"Expected {exception_type.__name__}, but no exception was raised")

print("Python:", sys.version.split()[0])
print("Test helper ready")

Python: 3.13.7
Test helper ready


---

## I · Exception fundamentals

Focus: legality of `raise`, exception payloads, validation contracts, and hierarchy.

### Problem 01 — Raise a class, an instance, and an invalid object

**Your task.** Write a function that accepts one of three modes: `'class'` raises the `ValueError` class, `'instance'` raises an existing `ValueError('bad input')`, and `'invalid'` attempts to raise an ordinary object. Predict the caught exception type in each case.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [2]:
class OrdinaryObject:
    pass


def raise_by_mode(mode):
    if mode == "class":
        # Python automatically constructs ValueError().
        raise ValueError
    if mode == "instance":
        original = ValueError("bad input")
        raise original
    if mode == "invalid":
        raise OrdinaryObject()
    raise ValueError(f"unknown mode: {mode}")

**Reasoning and important edge cases.** The operand may be an exception instance **or an exception class** derived from `BaseException`. An unrelated object fails with `TypeError`. Prefer instances with informative messages in production.

In [3]:
class_error = expect_exception(ValueError, lambda: raise_by_mode("class"))
instance_error = expect_exception(ValueError, lambda: raise_by_mode("instance"))
invalid_error = expect_exception(TypeError, lambda: raise_by_mode("invalid"))
assert class_error.args == ()
assert instance_error.args == ("bad input",)
assert "BaseException" in str(invalid_error)
assert isinstance(expect_exception(ValueError, lambda: raise_by_mode("typo")), ValueError)
print("Problem 01: PASS")

Problem 01: PASS


### Problem 02 — Design and inspect exception payloads

**Your task.** Create an exception with three positional payload values. Inspect `.args`, `str`, and `repr`. Contrast it with a single-message exception, and avoid hardcoding an implementation-specific traceback string.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [4]:
def build_error(*details):
    return ValueError(*details)


def describe_error(error):
    return {
        "class": type(error).__name__,
        "args": error.args,
        "as_text": str(error),
        "as_repr": repr(error),
    }

multi = describe_error(build_error("row", 7, "invalid"))
single = describe_error(build_error("invalid"))

**Reasoning and important edge cases.** `args` is a tuple. `str` of one argument is its string value; with multiple arguments it is the tuple's string representation for ordinary exceptions. Built-in subclasses may customize formatting, so write tests for the exception type you use.

In [5]:
assert multi["args"] == ("row", 7, "invalid")
assert multi["as_text"] == str(("row", 7, "invalid"))
assert multi["as_repr"] == "ValueError('row', 7, 'invalid')"
assert single["as_text"] == "invalid"
assert single["args"] == ("invalid",)
print("Problem 02: PASS")

Problem 02: PASS


### Problem 03 — Distinguish wrong type from wrong value

**Your task.** Implement `parse_percentage(value)`: only actual `int` or `float` values are permitted, explicitly reject `bool`; require 0–100 inclusive; return a normalized float. Use `TypeError` for an unacceptable *type*, `ValueError` for an unacceptable *value*. Reject NaN and infinity.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [6]:
import math


def parse_percentage(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError("percentage must be an int or float, not bool")
    if not math.isfinite(value):
        raise ValueError("percentage must be finite")
    if not 0 <= value <= 100:
        raise ValueError("percentage must be between 0 and 100")
    return float(value) / 100.0

**Reasoning and important edge cases.** `bool` is an `int` subclass, so a basic `isinstance(value, int)` accepts it. Reject it explicitly when the API means *numerical percentage*. Validating finiteness avoids NaN, which makes comparisons unexpectedly false.

In [7]:
assert parse_percentage(0) == 0.0
assert parse_percentage(12.5) == 0.125
assert parse_percentage(100) == 1.0
for bad in (True, "10", None, []):
    expect_exception(TypeError, lambda bad=bad: parse_percentage(bad))
for bad in (-1, 101, float("nan"), float("inf"), -float("inf")):
    expect_exception(ValueError, lambda bad=bad: parse_percentage(bad))
print("Problem 03: PASS")

Problem 03: PASS


### Problem 04 — Build a useful exception hierarchy

**Your task.** Model payment errors with `PaymentError` as a shared base, plus `CardDeclined` and `GatewayUnavailable`. Implement a handler that branches on specific errors before the broad parent. Explain why `Exception` is the recommended base for application failures.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [8]:
class PaymentError(Exception):
    """Base class for ordinary payment application failures."""


class CardDeclined(PaymentError):
    pass


class GatewayUnavailable(PaymentError):
    pass


def classify_payment_failure(error):
    if isinstance(error, CardDeclined):
        return "ask-for-another-card"
    if isinstance(error, GatewayUnavailable):
        return "retry-later"
    if isinstance(error, PaymentError):
        return "manual-review"
    raise TypeError("not a payment error")

**Reasoning and important edge cases.** Place specific branches before the general parent; a first-match `except PaymentError` would also catch all subclasses. `BaseException` includes `KeyboardInterrupt` and `SystemExit`; an application error should normally subclass `Exception`.

In [9]:
assert issubclass(CardDeclined, PaymentError)
assert issubclass(PaymentError, Exception)
assert classify_payment_failure(CardDeclined("declined")) == "ask-for-another-card"
assert classify_payment_failure(GatewayUnavailable("offline")) == "retry-later"
assert classify_payment_failure(PaymentError("other")) == "manual-review"
expect_exception(TypeError, lambda: classify_payment_failure(ValueError("wrong")))
print("Problem 04: PASS")

Problem 04: PASS


### Problem 05 — Preserve structured metadata in a custom exception

**Your task.** Define `RecordValidationError` carrying `row`, `field`, and `reason`. Keep a useful user-facing message, preserve machine-readable fields, and demonstrate that the inherited `.args` remains meaningful.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [10]:
class RecordValidationError(ValueError):
    def __init__(self, row, field, reason):
        self.row = row
        self.field = field
        self.reason = reason
        super().__init__(f"row {row}, field {field}: {reason}")


def validate_age(row, raw_age):
    if not isinstance(raw_age, int) or isinstance(raw_age, bool):
        raise RecordValidationError(row, "age", "integer required")
    if raw_age < 0:
        raise RecordValidationError(row, "age", "cannot be negative")
    return raw_age

**Reasoning and important edge cases.** A custom exception can expose structured fields without making consumers parse message text. Calling `super().__init__(message)` ensures useful exception `.args` and string formatting. This is an applied extension of the lesson's custom exception examples.

In [11]:
err = expect_exception(RecordValidationError, lambda: validate_age(8, -2))
assert (err.row, err.field, err.reason) == (8, "age", "cannot be negative")
assert err.args == ("row 8, field age: cannot be negative",)
assert validate_age(9, 0) == 0
expect_exception(RecordValidationError, lambda: validate_age(1, True))
print("Problem 05: PASS")

Problem 05: PASS


---

## II · Propagation, tracebacks and chaining

Focus: bare `raise`, explicit/implicit causes, suppression, and accurate traceback inspection.

### Problem 06 — Log then re-raise the *same* exception

**Your task.** Implement a division function that records a zero-division error in a supplied list and uses bare `raise` to propagate it. Prove the caller receives exactly the same object captured by the inner handler.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [12]:
def divide_with_audit(numerator, denominator, audit):
    try:
        return numerator // denominator
    except ZeroDivisionError as error:
        audit.append((type(error).__name__, error, error.args))
        raise  # Re-raise the currently handled exception.


audit_trail = []

**Reasoning and important edge cases.** Bare `raise` inside an active exception handler propagates the active exception and avoids adding a new explicit-raise site. Record only what you need; do not handle the failure silently.

In [13]:
assert divide_with_audit(8, 2, audit_trail) == 4
err = expect_exception(ZeroDivisionError, lambda: divide_with_audit(8, 0, audit_trail))
assert len(audit_trail) == 1
assert audit_trail[0][1] is err
assert audit_trail[0][0] == "ZeroDivisionError"
print("Problem 06: PASS")

Problem 06: PASS


### Problem 07 — Compare `raise` with `raise error`

**Your task.** Create two otherwise identical functions. In their handlers, use bare `raise` in one and `raise error` in the other. Compare traceback frame counts and explain the distinction. Do **not** rely on exact rendered traceback text.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [14]:
def propagate_using_bare_raise():
    try:
        raise LookupError("lookup failed")
    except LookupError:
        raise


def propagate_using_named_raise():
    try:
        raise LookupError("lookup failed")
    except LookupError as error:
        raise error


def traceback_functions(error):
    return [frame.name for frame in traceback.extract_tb(error.__traceback__)]

**Reasoning and important edge cases.** Both forms propagate the **same exception object** within their own handler. `raise error` executes a new raise at that line and therefore adds a traceback frame; bare `raise` is the usual choice to preserve propagation. Frame names are more stable than exact notebook display output.

In [15]:
bare = expect_exception(LookupError, propagate_using_bare_raise)
named = expect_exception(LookupError, propagate_using_named_raise)
assert traceback_functions(bare).count("propagate_using_bare_raise") == 1
assert traceback_functions(named).count("propagate_using_named_raise") == 2
assert str(bare) == str(named) == "lookup failed"
print("Problem 07: PASS")

Problem 07: PASS


### Problem 08 — Inspect implicit chaining precisely

**Your task.** Raise a `KeyError`, then a `RuntimeError` inside its handler *without* `from`. Inspect `__context__`, `__cause__` and `__suppress_context__` and find the characteristic renderer wording.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [16]:
def implicit_chain():
    try:
        raise KeyError("missing")
    except KeyError:
        raise RuntimeError("lookup service failed")


implicit = expect_exception(RuntimeError, implicit_chain)
implicit_rendered = "".join(traceback.format_exception(implicit))

**Reasoning and important edge cases.** Raising another exception while handling one creates an **implicit context**. The context is a separate exception object. `__cause__` remains `None`, and standard traceback formatting shows a “During handling ...” transition.

In [17]:
assert isinstance(implicit.__context__, KeyError)
assert implicit.__cause__ is None
assert implicit.__suppress_context__ is False
assert "During handling of the above exception" in implicit_rendered
assert "lookup service failed" in implicit_rendered
print("Problem 08: PASS")

Problem 08: PASS


### Problem 09 — Translate with an explicit causal link

**Your task.** Wrap a low-level `ZeroDivisionError` as a domain-specific `RateCalculationError` with `raise ... from error`. Inspect `__cause__` and the formatted chain; ensure successful calculations still work.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [18]:
class RateCalculationError(Exception):
    pass


def calculate_rate(total, count):
    try:
        return total / count
    except ZeroDivisionError as original:
        raise RateCalculationError("count must be nonzero") from original


chained = expect_exception(RateCalculationError, lambda: calculate_rate(5, 0))
chained_rendered = "".join(traceback.format_exception(chained))

**Reasoning and important edge cases.** Explicit `from error` designates the original exception as the **cause** and communicates a deliberate mapping across an abstraction boundary. It does *not* copy its traceback into the new exception; each exception has a traceback of its own.

In [19]:
assert calculate_rate(10, 2) == 5
assert isinstance(chained.__cause__, ZeroDivisionError)
assert chained.__suppress_context__ is True
assert "direct cause" in chained_rendered
assert "count must be nonzero" in chained_rendered
print("Problem 09: PASS")

Problem 09: PASS


### Problem 10 — Suppress display without deleting diagnostic context

**Your task.** Raise a public-facing `PublicLookupError` from `None` inside a caught `KeyError`. Verify what is displayed by `traceback.format_exception` and what remains available through attributes.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [20]:
class PublicLookupError(Exception):
    pass


def public_lookup(mapping, key):
    try:
        return mapping[key]
    except KeyError:
        raise PublicLookupError("requested item is unavailable") from None


public_error = expect_exception(PublicLookupError, lambda: public_lookup({}, "secret"))
public_rendered = "".join(traceback.format_exception(public_error))

**Reasoning and important edge cases.** Crucial nuance: `from None` sets `__suppress_context__` and suppresses **rendering** the original `KeyError`. The original remains accessible as `__context__`; the public error still has its own traceback. This corrects the source lesson's shorthand about “squashing the traceback.”

In [21]:
assert public_error.__cause__ is None
assert isinstance(public_error.__context__, KeyError)
assert public_error.__suppress_context__ is True
assert "KeyError" not in public_rendered
assert "PublicLookupError" in public_rendered
assert public_error.__traceback__ is not None
assert public_lookup({"ok": 42}, "ok") == 42
print("Problem 10: PASS")

Problem 10: PASS


### Problem 11 — Attach a traceback explicitly; do not confuse it with `from`

**Your task.** Catch an original error in a named function and create a new error with `.with_traceback(original.__traceback__) from original`. Inspect BOTH the new traceback frames and the explicit cause. Explain why this is rarely needed for an ordinary wrapper.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [22]:
class WrappedProcessingError(Exception):
    pass


def original_failure_point():
    raise OSError("device failed")


def attach_original_traceback():
    try:
        original_failure_point()
    except OSError as original:
        wrapped = WrappedProcessingError("processing failed")
        raise wrapped.with_traceback(original.__traceback__) from original


attached = expect_exception(WrappedProcessingError, attach_original_traceback)

**Reasoning and important edge cases.** `.with_traceback(tb)` alters the traceback associated with the **new** exception; `from original` sets its cause. The `raise` operation itself adds a new frame. Prefer a simple `raise Wrapped... from original` for most APIs; deliberately splicing tracebacks can confuse readers.

In [23]:
names = traceback_functions(attached)
assert "original_failure_point" in names
assert "attach_original_traceback" in names
assert isinstance(attached.__cause__, OSError)
assert attached.__traceback__ is not attached.__cause__.__traceback__
assert "original_failure_point" in traceback_functions(attached.__cause__)
print("Problem 11: PASS")

Problem 11: PASS


### Problem 12 — Create an exception-chain inspector

**Your task.** Write a safe function that walks explicit causes first, otherwise implicit contexts (unless suppressed). Detect cycles defensively. Return only exception type and message, not traceback text.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [24]:
def visible_exception_chain(error):
    chain = []
    seen = set()
    current = error
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        chain.append((type(current).__name__, str(current)))
        if current.__cause__ is not None:
            current = current.__cause__
        elif current.__suppress_context__:
            current = None
        else:
            current = current.__context__
    return chain

**Reasoning and important edge cases.** This models which link a normal formatter chooses: explicit cause takes precedence; a suppressed context is not displayed. It is not a replacement for `traceback.format_exception`, which also renders frame details. Cycle detection protects debugging tools from manually modified attributes.

In [25]:
assert [name for name, _ in visible_exception_chain(chained)] == [
    "RateCalculationError", "ZeroDivisionError"]
assert [name for name, _ in visible_exception_chain(implicit)] == [
    "RuntimeError", "KeyError"]
assert [name for name, _ in visible_exception_chain(public_error)] == [
    "PublicLookupError"]
cycle_a, cycle_b = ValueError("a"), ValueError("b")
cycle_a.__context__ = cycle_b
cycle_b.__context__ = cycle_a
assert len(visible_exception_chain(cycle_a)) == 2
print("Problem 12: PASS")

Problem 12: PASS


---

## III · Building reliable API boundaries

Focus: use appropriate exception contracts and preserve useful causes without exposing needless internal details.

### Problem 13 — Refactor a nested boolean converter

**Your task.** Reimplement the lesson's integer/string-to-boolean converter with a flat dispatch. Accept `bool`, integers 0/1, and strings 0/1/f/t/false/true case-insensitively. Reject floats and all other types. Raise one public `BooleanConversionError`, with a specific message for bad values versus unsupported types.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [26]:
class BooleanConversionError(ValueError):
    pass


def strict_bool(value):
    # Explicitly make bool handling a documented part of this API.
    if isinstance(value, bool):
        return value
    if type(value) is int:
        if value in (0, 1):
            return bool(value)
        raise BooleanConversionError("integer must be 0 or 1")
    if type(value) is str:
        normalized = value.casefold()
        if normalized in {"0", "f", "false"}:
            return False
        if normalized in {"1", "t", "true"}:
            return True
        raise BooleanConversionError("unrecognized boolean string")
    raise BooleanConversionError(f"unsupported type: {type(value).__name__}")

**Reasoning and important edge cases.** The source lesson solves this with nested `try`/`except TypeError` fallbacks. Explicit type dispatch is simpler when the input types are known. Using `type(value) is int` explicitly excludes custom subclasses; that is a deliberate API choice, not a universal rule. Here we avoid unnecessary exception chaining for plain validation.

In [27]:
truthy = [True, 1, "1", "T", "true", "TrUe"]
falsy = [False, 0, "0", "f", "FALSE"]
assert all(strict_bool(value) is True for value in truthy)
assert all(strict_bool(value) is False for value in falsy)
expect_exception(BooleanConversionError, lambda: strict_bool(2), contains="0 or 1")
expect_exception(BooleanConversionError, lambda: strict_bool("yes"), contains="string")
expect_exception(BooleanConversionError, lambda: strict_bool(1.0), contains="float")
expect_exception(BooleanConversionError, lambda: strict_bool([]), contains="list")
print("Problem 13: PASS")

Problem 13: PASS


### Problem 14 — Translate missing dictionary keys, but not every failure

**Your task.** Implement `required_field(payload, name)` to return a dictionary field, rejecting a non-dict payload using `TypeError`. Translate a missing key into `MissingFieldError` using explicit `from`, and prove it does not wrongly convert an unrelated type error.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [28]:
class MissingFieldError(LookupError):
    pass


def required_field(payload, name):
    if not isinstance(payload, dict):
        raise TypeError("payload must be a dict")
    try:
        return payload[name]
    except KeyError as original:
        raise MissingFieldError(f"required field missing: {name!r}") from original

**Reasoning and important edge cases.** Catch `KeyError` narrowly around the dictionary lookup. A blanket `except Exception` could hide programming errors and make debugging harder. The caller sees a stable domain error and can still inspect the original key failure through `__cause__`.

In [29]:
assert required_field({"id": 15}, "id") == 15
missing = expect_exception(MissingFieldError, lambda: required_field({}, "id"))
assert isinstance(missing.__cause__, KeyError)
assert missing.__cause__.args == ("id",)
expect_exception(TypeError, lambda: required_field(None, "id"))
print("Problem 14: PASS")

Problem 14: PASS


### Problem 15 — Translate only JSON decoding failures

**Your task.** Write `decode_object(source)` accepting JSON text that must decode to a dictionary. Translate `json.JSONDecodeError` into `InputFormatError` with an explicit cause and report a valid but non-object JSON value with `InputFormatError` but no fabricated cause.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [30]:
import json


class InputFormatError(ValueError):
    pass


def decode_object(source):
    if not isinstance(source, str):
        raise TypeError("JSON source must be text")
    try:
        decoded = json.loads(source)
    except json.JSONDecodeError as original:
        raise InputFormatError("invalid JSON document") from original
    if not isinstance(decoded, dict):
        raise InputFormatError("top-level JSON value must be an object")
    return decoded

**Reasoning and important edge cases.** A parser error and a schema error are different failures. Preserve the parser cause when there is one, but avoid implying there was a parser failure when the input was syntactically valid. Successful decoding is not wrapped in an exception.

In [31]:
assert decode_object('{"enabled": true}') == {"enabled": True}
syntax = expect_exception(InputFormatError, lambda: decode_object("{bad"))
assert isinstance(syntax.__cause__, json.JSONDecodeError)
shape = expect_exception(InputFormatError, lambda: decode_object("[1, 2]"))
assert shape.__cause__ is None
expect_exception(TypeError, lambda: decode_object(12))
print("Problem 15: PASS")

Problem 15: PASS


### Problem 16 — Build a public error with safe details

**Your task.** Imagine a secret-bearing config mapping. Return its `token` only when it is a nonempty string; otherwise expose a generic `ConfigurationError` without interpolating the secret into error messages. Suppress dictionary implementation details for a missing key.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [32]:
class ConfigurationError(Exception):
    pass


def read_secret_token(config):
    if not isinstance(config, dict):
        raise ConfigurationError("configuration is invalid")
    try:
        token = config["token"]
    except KeyError:
        raise ConfigurationError("required credential is unavailable") from None
    if not isinstance(token, str) or not token:
        raise ConfigurationError("required credential is invalid")
    return token

**Reasoning and important edge cases.** Do not put private values into messages, logging, or exception payloads. Suppression is a user-facing display choice; `__context__` is still inspectable, so it is **not** a security or sanitization boundary by itself. Limit access to full diagnostics where secrecy matters.

In [33]:
assert read_secret_token({"token": "example-token"}) == "example-token"
missing = expect_exception(ConfigurationError, lambda: read_secret_token({}))
assert missing.__suppress_context__
invalid = expect_exception(ConfigurationError, lambda: read_secret_token({"token": 12345}))
assert "12345" not in str(invalid)
expect_exception(ConfigurationError, lambda: read_secret_token([]))
print("Problem 16: PASS")

Problem 16: PASS


### Problem 17 — Preserve only the original failure across fallback attempts

**Your task.** First attempt to parse input as an integer; on failure, try a second parsing approach. If both approaches fail, raise a public `FallbackParseError` explicitly *from the first error*. Demonstrate what the explicit cause records; do not claim it magically rewrites or removes all intermediate tracebacks.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [34]:
class FallbackParseError(ValueError):
    pass


def int_with_fallback(value):
    try:
        return int(value)
    except (TypeError, ValueError) as first_error:
        try:
            return int(float(value))
        except (TypeError, ValueError, OverflowError):
            raise FallbackParseError("both integer parsing strategies failed") from first_error

**Reasoning and important edge cases.** `from first_error` selects the causal link (`__cause__`); it does not substitute the first exception's traceback for the new one's traceback. The second error can remain inspectable as implicit `__context__`, although an explicit cause takes precedence in formatted output. Using `int(float(value))` is intentionally only a fallback demo: it can lose precision on large numbers, so do not adopt it as a general financial parser.

In [35]:
assert int_with_fallback("17") == 17
assert int_with_fallback("17.8") == 17
fallback = expect_exception(FallbackParseError, lambda: int_with_fallback("not-a-number"))
assert isinstance(fallback.__cause__, ValueError)
assert isinstance(fallback.__context__, ValueError)
assert "direct cause" in "".join(traceback.format_exception(fallback))
print("Problem 17: PASS")

Problem 17: PASS


---

## IV · Applied extensions: cleanup, retries, logging, and robustness

These scenarios are additional practical applications beyond the original source lesson. All intentional failures are caught by tests.

### Problem 18 — Prove why `return` in `finally` is dangerous

**Your task.** Write a deliberately broken function that raises `ValueError` in `try` but returns from `finally`. Contrast with a corrected version that performs cleanup yet propagates the error. Explain why not to use `return` or `break` in cleanup blocks.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [36]:
def broken_finally():
    try:
        raise ValueError("important failure")
    finally:
        return "silent success"  # Deliberate anti-pattern.


def correct_finally(events):
    try:
        raise ValueError("important failure")
    finally:
        events.append("cleanup complete")  # No return or exception here.

**Reasoning and important edge cases.** A `return` inside `finally` overrides a pending exception, making a failure look successful. Likewise, exceptions thrown during cleanup can replace a primary exception. The correct form handles cleanup but lets the original error propagate.

In [37]:
assert broken_finally() == "silent success"
events = []
real_error = expect_exception(ValueError, lambda: correct_finally(events))
assert str(real_error) == "important failure"
assert events == ["cleanup complete"]
print("Problem 18: PASS")

Problem 18: PASS


### Problem 19 — Guarantee resource cleanup with a context manager

**Your task.** Implement a small context manager that records acquisition and release. It should release on both normal and exceptional exit and never swallow exceptions (`__exit__` must return `False`).

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [38]:
class TrackedResource:
    def __init__(self, events):
        self.events = events

    def __enter__(self):
        self.events.append("acquire")
        return self

    def __exit__(self, exc_type, exc, tb):
        self.events.append("release")
        return False  # Tell Python to propagate any pending exception.


def resource_operation(events, should_fail):
    with TrackedResource(events):
        events.append("work")
        if should_fail:
            raise OSError("work failed")
    return "ok"

**Reasoning and important edge cases.** Context managers make exception-safe cleanup explicit. Returning `True` from `__exit__` would suppress an exception. Avoid masking the original error with cleanup failures; more sophisticated resource managers may need to report both.

In [39]:
normal_events = []
assert resource_operation(normal_events, False) == "ok"
assert normal_events == ["acquire", "work", "release"]
failure_events = []
expect_exception(OSError, lambda: resource_operation(failure_events, True))
assert failure_events == ["acquire", "work", "release"]
print("Problem 19: PASS")

Problem 19: PASS


### Problem 20 — Log a failure without swallowing it

**Your task.** Instrument a function with `logging.exception` inside an `except OSError` block and then bare re-raise. Use a local in-memory handler so the exercise does not modify the notebook's global logger configuration.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [40]:
import logging


def load_with_logging(reader, logger):
    try:
        return reader()
    except OSError:
        logger.exception("read operation failed")
        raise


log_stream = StringIO()
exercise_logger = logging.getLogger("raising-exceptions.exercise20")
exercise_logger.handlers.clear()
exercise_logger.propagate = False
exercise_logger.setLevel(logging.ERROR)
handler = logging.StreamHandler(log_stream)
exercise_logger.addHandler(handler)

**Reasoning and important edge cases.** `logger.exception` includes traceback information when called from inside an active exception handler. Bare `raise` preserves the failure for the caller. Narrowly catch `OSError`; unrelated exceptions must not be inadvertently translated or logged as I/O failures.

In [41]:
def failing_reader():
    raise OSError("disk offline")

failure = expect_exception(OSError, lambda: load_with_logging(failing_reader, exercise_logger))
assert str(failure) == "disk offline"
assert "read operation failed" in log_stream.getvalue()
assert "OSError: disk offline" in log_stream.getvalue()
assert load_with_logging(lambda: "content", exercise_logger) == "content"
exercise_logger.removeHandler(handler)
handler.close()
print("Problem 20: PASS")

Problem 20: PASS


### Problem 21 — Retry only a transient exception and preserve exhaustion

**Your task.** Create `run_with_retries(operation, attempts)` that retries only `ConnectionError` and eventually re-raises the last such failure unchanged. Reject a non-positive retry count with `ValueError`. Propagate programming errors immediately.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [42]:
def run_with_retries(operation, attempts=3):
    if not isinstance(attempts, int) or isinstance(attempts, bool) or attempts < 1:
        raise ValueError("attempts must be a positive integer")
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except ConnectionError:
            if attempt == attempts:
                raise  # Final failure: same error, not an invented wrapper.
            # A real client should consider bounded backoff and idempotency.


def flaky_factory(failures):
    state = {"calls": 0}
    def operation():
        state["calls"] += 1
        if state["calls"] <= failures:
            raise ConnectionError(f"temporary outage #{state['calls']}")
        return "success"
    return operation, state

**Reasoning and important edge cases.** Retry only errors known to be transient, for bounded attempts, and only when repeating the operation is safe. Do not catch every exception or silently turn exhaustion into `None`. No sleeps or network dependencies are needed for this deterministic example.

In [43]:
temporary, state = flaky_factory(2)
assert run_with_retries(temporary, 3) == "success"
assert state["calls"] == 3
always_fails, state2 = flaky_factory(100)
exhausted = expect_exception(ConnectionError, lambda: run_with_retries(always_fails, 3))
assert "#3" in str(exhausted)
assert state2["calls"] == 3
programmer_calls = {"n": 0}
def bug():
    programmer_calls["n"] += 1
    raise TypeError("programming bug")
expect_exception(TypeError, lambda: run_with_retries(bug, 5))
assert programmer_calls["n"] == 1
expect_exception(ValueError, lambda: run_with_retries(lambda: None, 0))
print("Problem 21: PASS")

Problem 21: PASS


### Problem 22 — Model atomic rollback under an exception

**Your task.** Build a simple in-memory wallet transfer. Validate the amount, reject insufficient funds, and restore both balances if an unexpected failure happens **after** debit. Keep the original exception object, without wrapping it unnecessarily.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [44]:
class InsufficientFundsError(Exception):
    pass


def transfer_money(accounts, source, target, amount, *, fail_after_debit=False):
    if type(amount) is not int or amount <= 0:
        raise ValueError("amount must be a positive integer")
    if source not in accounts or target not in accounts:
        raise KeyError("account not found")
    if source == target:
        raise ValueError("accounts must be distinct")
    if accounts[source] < amount:
        raise InsufficientFundsError("not enough balance")
    source_before, target_before = accounts[source], accounts[target]
    try:
        accounts[source] -= amount
        if fail_after_debit:
            raise OSError("ledger write failed")
        accounts[target] += amount
    except Exception:
        accounts[source] = source_before
        accounts[target] = target_before
        raise
    return dict(accounts)

**Reasoning and important edge cases.** Rollback takes place only after the snapshot, and bare `raise` propagates the original failure. This toy implementation illustrates exception safety, not a real money-transfer design: production transactions need durable database transactions, concurrency controls, and idempotency.

In [45]:
balances = {"alice": 50, "bob": 10}
assert transfer_money(balances, "alice", "bob", 15) == {"alice": 35, "bob": 25}
before = dict(balances)
error = expect_exception(OSError, lambda: transfer_money(
    balances, "alice", "bob", 5, fail_after_debit=True))
assert str(error) == "ledger write failed"
assert balances == before
expect_exception(InsufficientFundsError, lambda: transfer_money(balances, "alice", "bob", 500))
assert balances == before
print("Problem 22: PASS")

Problem 22: PASS


---

## V · Advanced diagnostics and aggregated failures

Optional newer interpreter features are handled carefully; core content remains standard-library only.

### Problem 23 — Attach diagnostic notes without changing exception type

**Your task.** On Python 3.11+, add a note identifying a batch job to an existing `ValueError` and re-raise it. On older interpreters, leave the exception unchanged. Verify identity in either case.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [46]:
def annotate_and_propagate(job_name, error_log):
    try:
        raise ValueError("invalid record")
    except ValueError as error:
        error_log.append(error)
        if hasattr(error, "add_note"):
            error.add_note(f"job={job_name}")  # Available in Python 3.11+.
        raise


note_log = []
noted = expect_exception(ValueError, lambda: annotate_and_propagate("daily-import", note_log))

**Reasoning and important edge cases.** `BaseException.add_note` (Python 3.11+) appends diagnostic notes shown by normal traceback formatting; it does not replace `.args` or change the exception's class. Feature detection keeps the notebook runnable on Python 3.10.

In [47]:
assert noted is note_log[0]
assert noted.args == ("invalid record",)
if hasattr(noted, "add_note"):
    assert "job=daily-import" in noted.__notes__
    assert "job=daily-import" in "".join(traceback.format_exception(noted))
print("Problem 23: PASS")

Problem 23: PASS


### Problem 24 — Collect row failures instead of losing the first one

**Your task.** Validate a list of positive integers, collect a structured `RecordValidationError` per bad row, and raise an `AggregateValidationError` containing all failures. Verify that one bad row does not stop diagnosis of later rows.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [48]:
class AggregateValidationError(Exception):
    def __init__(self, errors):
        self.errors = tuple(errors)
        super().__init__(f"{len(self.errors)} row(s) failed validation")


def positive_rows(values):
    valid = []
    problems = []
    for row_number, value in enumerate(values, start=1):
        if type(value) is not int:
            problems.append(RecordValidationError(row_number, "value", "integer required"))
        elif value <= 0:
            problems.append(RecordValidationError(row_number, "value", "must be positive"))
        else:
            valid.append(value)
    if problems:
        raise AggregateValidationError(problems)
    return valid

**Reasoning and important edge cases.** Collecting validation errors improves user feedback when rows are independent. Keep structured exception objects rather than concatenating lossy strings. This exercise does not return partial successes: an invalid batch raises instead. `ExceptionGroup` is another option on Python 3.11+.

In [49]:
assert positive_rows([1, 2, 3]) == [1, 2, 3]
group = expect_exception(AggregateValidationError, lambda: positive_rows([2, 0, "3", -1, 5]))
assert len(group.errors) == 3
assert [error.row for error in group.errors] == [2, 3, 4]
assert all(isinstance(error, RecordValidationError) for error in group.errors)
assert group.args == ("3 row(s) failed validation",)
print("Problem 24: PASS")

Problem 24: PASS


### Problem 25 — Handle an ExceptionGroup selectively (Python 3.11+)

**Your task.** On supported Python, group a value error and a type error. Handle the value subgroup using `except* ValueError` while letting the type subgroup reach an outer ordinary handler. Do not use `except*` syntax directly in a code cell that must parse on Python 3.10.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [50]:
def exercise_exception_groups():
    if sys.version_info < (3, 11):
        return "skipped: Python < 3.11"
    # Compile only on 3.11+, so this notebook remains parseable on 3.10.
    namespace = {}
    example = """
def split_exception_group():
    handled = []
    try:
        try:
            raise ExceptionGroup('validation', [ValueError('bad value'), TypeError('bad type')])
        except* ValueError as matching:
            handled.extend(str(error) for error in matching.exceptions)
    except ExceptionGroup as remaining:
        return handled, tuple(type(error).__name__ for error in remaining.exceptions)
    raise AssertionError('The TypeError subgroup should have propagated')
"""
    exec(compile(example, "<exceptiongroup-exercise>", "exec"), namespace)
    return namespace["split_exception_group"]()

**Reasoning and important edge cases.** `except*` splits an exception group by matching types, and unmatched components propagate as a subgroup. Python 3.10 cannot even parse `except*`, so this exercise compiles the relevant source only when the running interpreter supports it. This is an extension beyond the provided lesson.

In [51]:
result = exercise_exception_groups()
if sys.version_info >= (3, 11):
    handled_messages, remaining_types = result
    assert handled_messages == ["bad value"]
    assert remaining_types == ("TypeError",)
else:
    assert result.startswith("skipped")
print("Problem 25: PASS")

Problem 25: PASS


---

## VI · Fully worked real-world capstones

Capstones integrate source concepts with applied validation, domain errors, causal links, and testable failure behavior.

### Problem 26 — Capstone A: atomic CSV order importer

**Your task.** Implement `import_orders(csv_text)`. A CSV header must have exactly `order_id,quantity,unit_price`. Each data row must have a unique nonempty `order_id`, a strictly positive integer quantity, and a nonnegative price with at most two decimal places. Return a list of fully validated order dictionaries. On any failure, raise `OrderImportError` with a row number and field. Preserve underlying `ValueError` or `decimal.InvalidOperation` causes where applicable. **Atomic:** do not return a partially validated batch.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [52]:
import csv
from decimal import Decimal, InvalidOperation


class OrderImportError(ValueError):
    def __init__(self, row, field, message):
        self.row = row
        self.field = field
        super().__init__(f"CSV row {row}, {field}: {message}")


def import_orders(csv_text):
    if not isinstance(csv_text, str):
        raise TypeError("csv_text must be a string")
    rows = csv.reader(StringIO(csv_text), strict=True)
    try:
        header = next(rows)
    except StopIteration:
        raise OrderImportError(1, "header", "missing header") from None
    except csv.Error as original:
        raise OrderImportError(1, "header", "invalid CSV") from original
    if header != ["order_id", "quantity", "unit_price"]:
        raise OrderImportError(1, "header", "unexpected column names")

    staged = []  # Only returned if all rows pass validation.
    seen_ids = set()
    try:
        for row_number, row in enumerate(rows, start=2):
            if len(row) != 3:
                raise OrderImportError(row_number, "row", "expected exactly 3 columns")
            order_id, quantity_raw, price_raw = row
            if not order_id:
                raise OrderImportError(row_number, "order_id", "must not be empty")
            if order_id in seen_ids:
                raise OrderImportError(row_number, "order_id", "duplicate order ID")
            seen_ids.add(order_id)
            try:
                quantity = int(quantity_raw)
            except ValueError as original:
                raise OrderImportError(row_number, "quantity", "invalid integer") from original
            if quantity <= 0 or quantity_raw.strip() != str(quantity):
                raise OrderImportError(row_number, "quantity", "positive canonical integer required")
            try:
                price = Decimal(price_raw)
            except InvalidOperation as original:
                raise OrderImportError(row_number, "unit_price", "invalid decimal") from original
            if (not price.is_finite() or price < 0
                    or price.as_tuple().exponent < -2):
                raise OrderImportError(row_number, "unit_price", "nonnegative, at most two decimals")
            staged.append({
                "order_id": order_id,
                "quantity": quantity,
                "unit_price": price,
                "line_total": quantity * price,
            })
    except csv.Error as original:
        raise OrderImportError(rows.line_num, "row", "invalid CSV") from original
    return staged

**Reasoning and important edge cases.** This importer validates **before** returning, preventing a partial result. The exception exposes row and field metadata; parser failures are translated with explicit causes, while schema failures are raised directly. `Decimal` avoids floating-point rounding for prices. `csv.reader` may accept some unusual quoting as valid CSV; `strict=True` enables stricter error handling but is not a full CSV schema validator. In production, define currency limits, encodings, file-size limits, duplicate normalization, and transaction semantics.

In [53]:
valid_csv = "order_id,quantity,unit_price\nA1,2,10.50\nB2,1,0.00\n"
orders = import_orders(valid_csv)
assert len(orders) == 2
assert orders[0]["line_total"] == Decimal("21.00")
assert orders[1]["unit_price"] == Decimal("0.00")
for bad_text, expected_row, expected_field in [
    ("", 1, "header"),
    ("id,qty,price\n", 1, "header"),
    ("order_id,quantity,unit_price\nA1,0,10.00\n", 2, "quantity"),
    ("order_id,quantity,unit_price\nA1,2,10.00\nA1,3,8.00\n", 3, "order_id"),
    ("order_id,quantity,unit_price\nA1,x,5.00\n", 2, "quantity"),
    ("order_id,quantity,unit_price\nA1,1,NaN\n", 2, "unit_price"),
    ("order_id,quantity,unit_price\nA1,1,1.234\n", 2, "unit_price"),
]:
    issue = expect_exception(OrderImportError, lambda text=bad_text: import_orders(text))
    assert (issue.row, issue.field) == (expected_row, expected_field)
parse_failure = expect_exception(OrderImportError, lambda: import_orders(
    "order_id,quantity,unit_price\nA1,x,2.00\n"))
assert isinstance(parse_failure.__cause__, ValueError)
expect_exception(TypeError, lambda: import_orders(None))
print("Problem 26: PASS")

Problem 26: PASS


### Problem 27 — Capstone B: configuration loading with error boundaries

**Your task.** Implement a two-layer configuration API: `load_raw_config(text)` parses JSON to a dict; `get_timeout(config)` checks that `timeout_ms` is an actual integer between 1 and 60,000; `initialize_service(text)` offers a single public `ServiceStartupError` with an explicit cause for expected config errors. Unexpected programming errors must not be swallowed. Include tests for syntax, missing field, wrong type, out of range, and success.

**Success criteria.** The reference implementation must pass the assertions below; intentional failures must be caught so the notebook runs from top to bottom.

**Worked solution**

In [54]:
class ServiceStartupError(Exception):
    pass


class ConfigSchemaError(ValueError):
    pass


def load_raw_config(text):
    if not isinstance(text, str):
        raise TypeError("config source must be a string")
    try:
        config = json.loads(text)
    except json.JSONDecodeError as original:
        raise ConfigSchemaError("config syntax is invalid") from original
    if not isinstance(config, dict):
        raise ConfigSchemaError("config must be an object")
    return config


def get_timeout(config):
    try:
        timeout = config["timeout_ms"]
    except KeyError as original:
        raise ConfigSchemaError("timeout_ms is required") from original
    if type(timeout) is not int:
        raise ConfigSchemaError("timeout_ms must be an integer")
    if not 1 <= timeout <= 60_000:
        raise ConfigSchemaError("timeout_ms is outside 1..60000")
    return timeout


def initialize_service(text):
    try:
        config = load_raw_config(text)
        timeout = get_timeout(config)
    except ConfigSchemaError as original:
        raise ServiceStartupError("service configuration rejected") from original
    return {"status": "ready", "timeout_ms": timeout}

**Reasoning and important edge cases.** Each layer owns its exception contract: JSON decoding maps to schema failure, domain initialization maps schema failure to startup failure. A chain such as `ServiceStartupError → ConfigSchemaError → JSONDecodeError` explains *why* a startup failed. An incompatible source type raises `TypeError` unchanged because the public API documents it as a programming error.

In [55]:
assert initialize_service('{"timeout_ms": 5000}') == {"status": "ready", "timeout_ms": 5000}
for invalid_text in ('{bad', '{}', '{"timeout_ms": true}',
                     '{"timeout_ms": 0}', '{"timeout_ms": 60001}', '[1]'):
    startup = expect_exception(ServiceStartupError, lambda text=invalid_text: initialize_service(text))
    assert isinstance(startup.__cause__, ConfigSchemaError)
syntax_error = expect_exception(ServiceStartupError, lambda: initialize_service('{bad'))
assert isinstance(syntax_error.__cause__.__cause__, json.JSONDecodeError)
expect_exception(TypeError, lambda: initialize_service(None))
print("Problem 27: PASS")

Problem 27: PASS


---

## Final mastery checklist and further challenges

- Can you distinguish **exception class** and **exception instance** operands to `raise`?
- Can you explain why a bare `raise` differs from `raise error` when inspecting frames?
- Can you distinguish `__cause__`, `__context__`, `__suppress_context__`, and `__traceback__` without describing them as interchangeable?
- Can you explain why `from None` is a display choice, **not** destruction of hidden context or a privacy guarantee?
- Does every custom exception use an appropriate base class and preserve a useful `.args` value?
- Are caught exception types as narrow as reasonable? Are `KeyboardInterrupt` and `SystemExit` left alone?
- Are cleanup paths exception safe? Do retry policies have finite bounds and appropriate failure types?
- Do importers validate a whole batch before committing or returning a partial result?

**Independent stretch tasks (without supplied solutions):** extend the CSV importer to collect all row errors; add a maximum error count; implement an in-memory database transaction that preserves both the commit and rollback failures; add property-based tests if you choose to install a testing library. These are optional beyond the 27 solved problems.

**Accuracy note:** The lesson's examples are used as the topic and conceptual foundation; the problems and new applications here are newly created. The corrections in the introduction reflect Python's actual exception semantics, which are directly tested in the notebook.